# GreenMind AI: Notebook 2, train species classifiers on the iNaturalist dataset

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**.
**Input:** add the output of the notebook that holds `inat.tar` (Add Input → Notebook → `greenmind-ai-notebook-2`).
If no `inat.tar` is attached, the images are downloaded first (about 6 h extra).
Then **Save Version → Save & Run All (Commit)**, about 4–5 h with `inat.tar` attached.

Trains ResNet-18 (same architecture as the app) and EfficientNet-B0 on a balanced subset of `pool_a`
(target 139,000 images, 70/15/15 split), then tests both on the in-distribution test split and on the temporal hold-out `pool_b`.
The deployed app model (saved Sept 2025) is also tested on `pool_b`.

In [ ]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("greenmind-ai"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mubashir20023/greenmind-ai"], check=True)
os.chdir("/kaggle/working/greenmind-ai")
assert os.path.exists("tools/inat_fetch_species.py"), "Push the new tools/ folder to GitHub first (see RUN_EXPERIMENTS.md, Step 0)"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==1.0.19", "scikit-learn"], check=False)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
import glob, os, subprocess
if not os.path.exists("/kaggle/working/greenmind-ai"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/mubashir20023/greenmind-ai", "/kaggle/working/greenmind-ai"], check=True)
os.chdir("/kaggle/working/greenmind-ai")
os.makedirs("/kaggle/temp", exist_ok=True)

c = glob.glob("/kaggle/input/**/inat/fetch_stats.json", recursive=True)
t = glob.glob("/kaggle/input/**/inat.tar", recursive=True)
if c:
    INAT = os.path.dirname(c[0])
elif t:
    print("extracting", t[0])
    !tar -xf "{t[0]}" -C /kaggle/temp
    INAT = "/kaggle/temp/inat"
else:
    # download the iNaturalist images (about 6 h), then keep ONE archive file so Kaggle saves it
    !python tools/inat_fetch_species.py --out /kaggle/temp/inat --per_class_a 150 --per_class_b 20 --workers 16
    !tar -cf /kaggle/working/inat.tar -C /kaggle/temp inat
    INAT = "/kaggle/temp/inat"
print(INAT)
!cat {INAT}/fetch_stats.json
!ls {INAT}/pool_a | wc -l

In [ ]:
# Deployed model (Sept 2025) on the temporal hold-out it has never seen
import os, urllib.request
ck = "models/checkpoints/plant_id_resnet18_best.pth"
if os.path.getsize(ck) < 10_000:
    urllib.request.urlretrieve("https://media.githubusercontent.com/media/mubashir20023/greenmind-ai/main/" + ck, ck)
!python tools/evaluate.py --ckpt {ck} --arch resnet18 --classes models/class_maps/idx_to_species.json \
    --imagefolder {INAT}/pool_b --workers 4 --out /kaggle/working/results/e1_deployed_resnet18_temporal

In [ ]:
!python tools/make_split.py --src {INAT}/pool_a --total 139000 --min_per_class 30 --val 0.15 --test 0.15 \
    --out_dir /kaggle/working/split139k

In [ ]:
!python tools/train_classifier.py --manifest /kaggle/working/split139k/manifest.csv --arch resnet18 --norm imagenet \
    --epochs 20 --batch_size 128 --lr 1e-3 --workers 4 --out /kaggle/working/species_resnet18

In [ ]:
!python tools/evaluate.py --ckpt /kaggle/working/species_resnet18/best.pth --arch resnet18 \
    --classes /kaggle/working/species_resnet18/classes.json --imagefolder {INAT}/pool_b --workers 4 \
    --out /kaggle/working/results/e2_resnet18_temporal

In [ ]:
!python tools/train_classifier.py --manifest /kaggle/working/split139k/manifest.csv --arch efficientnet_b0 --norm imagenet \
    --epochs 20 --batch_size 128 --lr 1e-3 --workers 4 --out /kaggle/working/species_effb0

In [ ]:
!python tools/evaluate.py --ckpt /kaggle/working/species_effb0/best.pth --arch efficientnet_b0 \
    --classes /kaggle/working/species_effb0/classes.json --imagefolder {INAT}/pool_b --workers 4 \
    --out /kaggle/working/results/e2_effb0_temporal

In [ ]:
!rm -f /kaggle/working/*/last.pth
!cat /kaggle/working/split139k/split_stats.json
!for f in /kaggle/working/species_*/eval_test/metrics.json /kaggle/working/results/*/metrics.json; do echo $f; head -12 $f; done